# 01 EDA Data Sintetis ProcureWise

Notebook ini adalah tahap **Data Understanding**. Data yang dipakai adalah output generator dengan seed 15037 di `ml/data/raw/`. Cara data dibuat dijelaskan di `ml/README.md`.

Isi notebook:

1. Distribusi harga per material dan per kategori
2. On-time rate dan defect rate per vendor
3. Harga vs kualitas tersembunyi vendor
4. Efek musim ramai ke keterlambatan
5. Sebaran grade label dan kandidat yang gugur filter keras

Catatan penting: file berawalan `_` (karakter tersembunyi vendor dan label) dipakai di sini **hanya untuk memahami data**. Karakter tersembunyi tidak boleh menjadi fitur model.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# grafik tampil di vs code, jupyterlab, dan nbviewer
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
pd.set_option("display.precision", 3)

CATEGORY_COLORS = {"bahan": "#2a6fdb", "jasa": "#e07b39", "packaging": "#3c9d5d"}


def find_data_dir() -> Path:
    # notebook bisa dijalankan dari ml/notebooks, ml, atau root repo
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base / "data" / "raw", base / "ml" / "data" / "raw"):
            if (candidate / "_manifest.json").exists():
                return candidate
    raise FileNotFoundError("data/raw belum ada, jalankan dulu: python -m generator")


DATA_DIR = find_data_dir()
manifest = json.loads((DATA_DIR / "_manifest.json").read_text(encoding="utf-8"))
assert manifest["seed"] == 15037, "notebook ini disusun untuk data seed 15037"

materials = pd.read_csv(DATA_DIR / "materials.csv")
vendors = pd.read_csv(DATA_DIR / "vendors.csv")
offers = pd.read_csv(DATA_DIR / "vendor_offers.csv")
requests = pd.read_csv(DATA_DIR / "procurement_requests.csv", parse_dates=["request_date", "needed_by"])
tx = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["order_date", "promised_date", "delivered_date"])
candidates = pd.read_csv(DATA_DIR / "_candidates.csv")
labels = pd.read_csv(DATA_DIR / "_labels.csv")
meta = pd.read_csv(DATA_DIR / "_request_meta.csv")
traits = pd.read_csv(DATA_DIR / "_hidden_vendor_traits.csv")

print(f"seed {manifest['seed']}, config hash {manifest['config_hash']}")
pd.DataFrame({"baris": manifest["rows"]})

seed 15037, config hash 7b699cf7fe6d


,baris
users.csv,2
materials.csv,15
vendors.csv,28
vendor_offers.csv,77
procurement_requests.csv,2102
transactions.csv,2050
recommendations.csv,0
_hidden_vendor_traits.csv,28
_labels.csv,7191
_candidates.csv,9754


## 1. Distribusi Harga per Material dan per Kategori

Harga diambil dari snapshot penawaran di `_candidates.csv`, yaitu harga setiap vendor pada saat request dibuat. Karena skala harga antar material sangat berbeda (Rp200 sampai puluhan ribu), sumbu harga memakai skala log.

In [2]:
cand = candidates.merge(requests[["id", "material_id", "request_date"]], left_on="request_id", right_on="id")
cand = cand.merge(materials[["id", "code", "category", "unit"]].rename(columns={"id": "material_id"}), on="material_id")
material_order = materials["code"].tolist()

fig = px.box(
    cand, x="code", y="unit_price", color="category", color_discrete_map=CATEGORY_COLORS,
    category_orders={"code": material_order}, log_y=True, points=False,
    labels={"code": "Material", "unit_price": "Harga per satuan (Rp, skala log)", "category": "Kategori"},
    title="Harga penawaran per material saat request",
)
fig.update_layout(height=450)
fig.show()

price_table = (
    cand.groupby(["code", "unit"])["unit_price"]
    .agg(n="size", median="median", p10=lambda s: s.quantile(0.1), p90=lambda s: s.quantile(0.9))
    .reindex(material_order, level=0)
)
price_table["p90/p10"] = price_table["p90"] / price_table["p10"]
price_table.round(1)

,,n,median,p10,p90,p90/p10
code,unit,,,,,
B01,pcs,912,35150.0,31000.0,44200.0,1.4
B02,pcs,1218,5300.0,4850.0,7600.0,1.6
B03,pcs,1048,51000.0,36200.0,57800.0,1.6
B04,pcs,642,28500.0,19810.0,32100.0,1.6
B05,pcs,456,5050.0,4300.0,6200.0,1.4
J01,pcs,872,6450.0,5600.0,7500.0,1.3
J02,pcs,436,11300.0,9700.0,12500.0,1.3
J03,pcs,416,9600.0,8500.0,10900.0,1.3
J04,lembar,530,1600.0,1100.0,1900.0,1.7


**Temuan:** Sebaran harga antar vendor untuk satu material cukup sempit, rasio p90/p10 ada di kisaran 1,3 sampai 2,0 kali. Rasio terbesar muncul di material murah seperti stiker label (J05) dan kartu ucapan (J04), sebagian karena pembulatan harga ke Rp50 membuat selisih kecil terlihat besar.

In [3]:
# harga relatif = harga vendor dibagi median harga semua kandidat di request yang sama
cand["rel_price"] = cand["unit_price"] / cand.groupby("request_id")["unit_price"].transform("median")

fig = px.violin(
    cand, x="category", y="rel_price", color="category", color_discrete_map=CATEGORY_COLORS, box=True,
    points=False, labels={"category": "Kategori", "rel_price": "Harga relatif terhadap median kandidat"},
    title="Harga relatif vendor dalam satu request, per kategori",
)
fig.add_hline(y=1, line_dash="dot", line_color="gray")
fig.update_layout(height=420, showlegend=False)
fig.show()

cand.groupby("category")["rel_price"].describe(percentiles=[0.1, 0.5, 0.9]).round(3)

,count,mean,std,min,10%,50%,90%,max
category,,,,,,,,
bahan,4276.0,1.000,0.173,0.673,0.719,1.0,1.245,1.450
jasa,2565.0,0.992,0.140,0.667,0.854,1.0,1.156,1.333
packaging,2913.0,1.009,0.109,0.810,0.873,1.0,1.146,1.270


**Temuan:** Kategori bahan punya variasi harga relatif paling lebar (10% termurah di bawah 0,72 kali median dan 10% termahal di atas 1,24 kali median), sedangkan packaging paling rapat (0,87 sampai 1,15 kali median). Jadi pilihan vendor paling berpengaruh ke total biaya di kategori bahan.

## 2. On-time Rate dan Defect Rate per Vendor

On-time dihitung dari transaksi `completed` dengan `delivered_date <= promised_date`. Defect rate adalah total `qty_defect` dibagi total `qty`. Vendor 6 belum punya transaksi selesai karena baru bergabung di akhir periode (cold start).

In [4]:
done = tx[tx["status"] == "completed"].copy()
done["on_time"] = done["delivered_date"] <= done["promised_date"]
done["delay_days"] = (done["delivered_date"] - done["promised_date"]).dt.days.clip(lower=0)

vendor_perf = (
    done.groupby("vendor_id")
    .agg(n_selesai=("id", "size"), on_time_rate=("on_time", "mean"), avg_delay=("delay_days", "mean"),
         qty_defect=("qty_defect", "sum"), qty=("qty", "sum"), avg_rating=("quality_rating", "mean"))
    .assign(defect_rate=lambda d: d["qty_defect"] / d["qty"])
    .drop(columns=["qty_defect", "qty"])
)
vendor_perf = vendors[["id", "name", "category"]].merge(vendor_perf, left_on="id", right_index=True)

fig = px.scatter(
    vendor_perf, x="on_time_rate", y="defect_rate", size="n_selesai", color="category",
    color_discrete_map=CATEGORY_COLORS, text="id", hover_name="name", size_max=40,
    labels={"on_time_rate": "On-time rate", "defect_rate": "Defect rate", "category": "Kategori",
            "n_selesai": "Transaksi selesai"},
    title="On-time rate vs defect rate per vendor (ukuran titik = jumlah transaksi selesai)",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=520, xaxis_tickformat=".0%", yaxis_tickformat=".1%")
fig.show()

vendor_perf.sort_values("on_time_rate").set_index("id").round(3)

,name,category,n_selesai,on_time_rate,avg_delay,avg_rating,defect_rate
id,,,,,,,
19,PT Karya Lestari Digital Print,jasa,11,0.455,1.273,3.364,0.014
4,UD Abadi Cahaya Tekstil,bahan,68,0.456,2.206,4.103,0.010
15,Toko Insan Prima Printing,jasa,6,0.500,0.500,4.333,0.001
3,UD Sentosa Sumber Tekstil,bahan,168,0.524,2.036,2.054,0.071
12,Toko Mulia Sentosa Embroidery,jasa,16,0.562,3.562,3.375,0.020
7,PT Mega Barokah Grosir Muslim,bahan,28,0.571,1.964,2.821,0.037
22,Toko Mega Makmur Kemasan,packaging,38,0.579,1.895,3.026,0.020
28,PT Prima Sumber Grosir Kemasan,packaging,10,0.600,0.400,3.400,0.019
13,CV Harapan Duta Embroidery,jasa,195,0.631,1.585,2.056,0.088


**Temuan:** On-time rate antar vendor berkisar dari sekitar 46% sampai 94% dan defect rate dari di bawah 1% sampai 8,8%. Vendor 3 dan 13 termasuk yang paling sering dipakai (168 dan 195 transaksi selesai) padahal defect rate-nya paling tinggi, sehingga histori berisi banyak contoh vendor murah yang bermasalah.

## 3. Harga vs Kualitas Tersembunyi Vendor

Sumbu x adalah `price_level` (pengali harga vendor terhadap harga referensi) dan sumbu y adalah `true_quality`. Keduanya karakter tersembunyi dari `_hidden_vendor_traits.csv`. Warna titik menunjukkan on-time rate yang **teramati** di transaksi.

Kelompok yang ditandai (hanya vendor dengan minimal 10 transaksi selesai):

- **Murah tapi sering telat**: `price_level` < 0,95 dan on-time rate < 70%
- **Mahal tapi cepat**: `price_level` > 1,05 dan on-time rate >= 85%

In [5]:
vendor_view = traits.merge(vendor_perf, left_on="vendor_id", right_on="id", how="left")
enough = vendor_view["n_selesai"] >= 10
cheap_late = enough & (vendor_view["price_level"] < 0.95) & (vendor_view["on_time_rate"] < 0.70)
pricey_fast = enough & (vendor_view["price_level"] > 1.05) & (vendor_view["on_time_rate"] >= 0.85)
vendor_view["kelompok"] = np.select(
    [cheap_late, pricey_fast], ["Murah tapi sering telat", "Mahal tapi cepat"], default="Lainnya"
)

symbols = {"Murah tapi sering telat": "x", "Mahal tapi cepat": "diamond", "Lainnya": "circle"}
fig = go.Figure()
for group, symbol in symbols.items():
    part = vendor_view[vendor_view["kelompok"] == group]
    fig.add_trace(go.Scatter(
        x=part["price_level"], y=part["true_quality"], mode="markers+text", name=group,
        text=part["vendor_id"], textposition="top center",
        marker=dict(symbol=symbol, size=14, color=part["on_time_rate"], colorscale="RdYlGn", cmin=0.4, cmax=1.0,
                    line=dict(width=1, color="black"),
                    colorbar=dict(title="On-time", tickformat=".0%") if group == "Lainnya" else None),
        customdata=part[["true_reliability", "n_selesai", "archetype"]].fillna("-"),
        hovertemplate="vendor %{text}<br>price_level %{x:.2f}<br>true_quality %{y:.2f}"
                      "<br>true_reliability %{customdata[0]:.2f}<br>transaksi selesai %{customdata[1]}"
                      "<br>arketipe %{customdata[2]}<extra></extra>",
    ))
fig.add_vline(x=1, line_dash="dot", line_color="gray")
fig.update_layout(
    title="Harga vs kualitas tersembunyi vendor, warna = on-time rate teramati",
    xaxis_title="price_level (tersembunyi)", yaxis_title="true_quality (tersembunyi)", height=560,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()

corr = vendor_view[["price_level", "true_quality", "true_reliability"]].corr().round(2)
print("korelasi antar karakter tersembunyi")
print(corr)
vendor_view.loc[vendor_view["kelompok"] != "Lainnya",
                ["vendor_id", "category", "kelompok", "price_level", "true_quality", "true_reliability",
                 "n_selesai", "on_time_rate", "defect_rate"]].sort_values(["kelompok", "price_level"]).round(2)

korelasi antar karakter tersembunyi
                  price_level  true_quality  true_reliability
price_level              1.00          0.44              0.02
true_quality             0.44          1.00              0.09
true_reliability         0.02          0.09              1.00


,vendor_id,category,kelompok,price_level,true_quality,true_reliability,n_selesai,on_time_rate,defect_rate
24,25,packaging,Mahal tapi cepat,1.10,0.76,0.70,171.0,0.88,0.01
9,10,bahan,Mahal tapi cepat,1.23,0.42,0.78,11.0,0.91,0.03
26,27,packaging,Mahal tapi cepat,1.27,0.76,0.92,16.0,0.94,0.01
12,13,jasa,Murah tapi sering telat,0.82,0.41,0.43,195.0,0.63,0.09
2,3,bahan,Murah tapi sering telat,0.83,0.36,0.31,168.0,0.52,0.07
22,23,packaging,Murah tapi sering telat,0.88,0.39,0.45,135.0,0.64,0.05
3,4,bahan,Murah tapi sering telat,0.92,0.88,0.38,68.0,0.46,0.01


**Temuan:** Harga berkorelasi positif dengan kualitas (0,44) tapi hampir tidak berkorelasi dengan keandalan (0,02), jadi harga mahal tidak menjamin tepat waktu. Ada empat vendor murah tapi sering telat (vendor 3, 4, 13, dan 23 dengan on-time 46% sampai 64%) dan tiga vendor mahal tapi cepat (vendor 10, 25, dan 27 dengan on-time 88% sampai 94%), termasuk vendor 4 yang kualitasnya tinggi (0,88) tapi on-time hanya 46%.

## 4. Efek Musim Ramai ke Keterlambatan

Intensitas musim ramai `s(t)` diambil dari `_request_meta.csv` (nilai 0 sampai 1 pada tanggal order). Transaksi dikelompokkan menjadi Normal (s = 0), Ringan (0 < s < 0,5), Ramai (0,5 <= s < 0,8), dan Puncak (s >= 0,8).

In [6]:
season_tx = done.merge(meta[["request_id", "season_intensity"]], on="request_id")
season_tx["musim"] = pd.cut(
    season_tx["season_intensity"], bins=[-0.01, 0.0, 0.499, 0.799, 1.0],
    labels=["Normal", "Ringan", "Ramai", "Puncak"],
)
season_table = season_tx.groupby("musim", observed=True).agg(
    n=("id", "size"), on_time_rate=("on_time", "mean"), avg_delay=("delay_days", "mean"),
    p_telat_7_hari=("delay_days", lambda s: (s >= 7).mean()),
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=season_table.index.astype(str), y=season_table["on_time_rate"], name="On-time rate",
            marker_color="#2a6fdb", text=season_table["on_time_rate"].map("{:.0%}".format), textposition="outside")
fig.add_scatter(x=season_table.index.astype(str), y=season_table["avg_delay"], name="Rata-rata telat (hari)",
                mode="lines+markers", line=dict(color="#d64545", width=3), secondary_y=True)
fig.update_yaxes(title_text="On-time rate", tickformat=".0%", range=[0, 1.05], secondary_y=False)
fig.update_yaxes(title_text="Rata-rata telat (hari)", secondary_y=True, rangemode="tozero")
fig.update_layout(title="On-time rate dan keterlambatan per tingkat musim ramai", height=430,
                  legend=dict(orientation="h", y=-0.15))
fig.show()
season_table.round(3)

,n,on_time_rate,avg_delay,p_telat_7_hari
musim,,,,
Normal,621,0.834,0.675,0.037
Ringan,267,0.798,0.858,0.041
Ramai,815,0.747,0.983,0.047
Puncak,260,0.512,2.977,0.177


**Temuan:** On-time rate turun dari 83% di musim normal menjadi 51% saat puncak, dan proporsi pesanan yang telat 7 hari atau lebih naik dari 3,7% menjadi 17,7%. Penyebabnya, janji lead time vendor hanya naik 15% saat ramai, sementara lead time aslinya naik sampai 35% dan kapasitas efektifnya turun.

In [7]:
monthly = season_tx.assign(bulan=season_tx["order_date"].dt.to_period("M").dt.to_timestamp()).groupby("bulan").agg(
    n=("id", "size"), on_time_rate=("on_time", "mean"), avg_delay=("delay_days", "mean"),
    season=("season_intensity", "mean"),
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=monthly.index, y=monthly["season"], name="Rata-rata s(t)", marker_color="rgba(224,123,57,0.35)",
            secondary_y=True)
fig.add_scatter(x=monthly.index, y=monthly["on_time_rate"], name="On-time rate", mode="lines+markers",
                line=dict(color="#2a6fdb", width=3))
fig.update_yaxes(title_text="On-time rate", tickformat=".0%", range=[0.4, 1.0], secondary_y=False)
fig.update_yaxes(title_text="Rata-rata s(t)", range=[0, 1.2], secondary_y=True)
fig.update_layout(title="On-time rate bulanan dan intensitas musim ramai (berdasarkan tanggal order)", height=430,
                  legend=dict(orientation="h", y=-0.15))
fig.show()
monthly.round(3)

,n,on_time_rate,avg_delay,season
bulan,,,,
2025-04-01,111,0.667,1.333,0.500
2025-05-01,131,0.756,0.962,0.485
2025-06-01,107,0.776,0.822,0.359
2025-07-01,80,0.850,1.088,0.000
2025-08-01,86,0.884,0.605,0.000
2025-09-01,81,0.840,0.667,0.000
2025-10-01,79,0.886,0.291,0.000
2025-11-01,138,0.754,1.123,0.600
2025-12-01,142,0.803,0.620,0.465


**Temuan:** Titik terendah ada di Maret 2026 (on-time 39% dengan rata-rata telat 4,6 hari), yaitu order di puncak Ramadan yang pengerjaannya terpotong libur Lebaran. Di bulan normal seperti Juli sampai Oktober 2025, on-time stabil di 84% sampai 89%.

## 5. Sebaran Grade Label dan Kandidat yang Gugur Filter

Grade 0 sampai 3 di `_labels.csv` hanya ada untuk kandidat yang lolos filter keras. Grade dihitung dari simulasi hasil pesanan (lihat `ml/README.md` bagian 13), bukan dari fitur.

In [8]:
lab = labels.merge(meta[["request_id", "urgency", "split", "is_training_group"]], on="request_id")
lab = lab.merge(requests[["id", "priority"]], left_on="request_id", right_on="id")

grade_overall = lab["grade"].value_counts(normalize=True).sort_index()
grade_by_urgency = pd.crosstab(lab["urgency"], lab["grade"], normalize="index").reindex(["ketat", "normal", "santai"])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Semua kandidat lolos filter", "Per tingkat urgensi request"),
                    column_widths=[0.4, 0.6])
grade_colors = {0: "#d64545", 1: "#f0a35e", 2: "#8cc084", 3: "#2e8b57"}
fig.add_bar(x=[f"grade {g}" for g in grade_overall.index], y=grade_overall.values,
            marker_color=[grade_colors[g] for g in grade_overall.index],
            text=[f"{v:.1%}" for v in grade_overall.values], textposition="outside", showlegend=False, row=1, col=1)
for g in grade_by_urgency.columns:
    fig.add_bar(x=grade_by_urgency.index, y=grade_by_urgency[g], name=f"grade {g}", marker_color=grade_colors[g],
                row=1, col=2)
fig.update_layout(barmode="stack", title="Sebaran grade label", height=430, legend=dict(orientation="h", y=-0.15))
fig.update_yaxes(tickformat=".0%", row=1, col=1, range=[0, 0.5])
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.show()

print("proporsi grade per urgensi")
grade_by_urgency.round(3)

proporsi grade per urgensi


grade,0,1,2,3
urgency,,,,
ketat,0.419,0.228,0.258,0.095
normal,0.141,0.299,0.395,0.166
santai,0.082,0.220,0.459,0.239


In [9]:
group_stats = lab[lab["is_training_group"]].groupby("request_id")["grade"].agg(["size", "nunique", "max"])
print(f"grup training: {len(group_stats):,}")
print(f"rata-rata kandidat lolos per grup: {group_stats['size'].mean():.2f}")
print(f"grup dengan semua grade sama: {(group_stats['nunique'] == 1).mean():.1%}")
print(f"grup tanpa satu pun kandidat grade 3: {(group_stats['max'] < 3).mean():.1%}")
print(f"grup yang semua kandidatnya grade 0: {(group_stats['max'] == 0).mean():.1%}")

grup training: 1,942
rata-rata kandidat lolos per grup: 3.65
grup dengan semua grade sama: 14.1%
grup tanpa satu pun kandidat grade 3: 51.7%
grup yang semua kandidatnya grade 0: 5.4%


**Temuan:** Secara keseluruhan sebaran grade cukup seimbang (grade 0 19% dan grade 3 17%), tapi sangat bergantung pada urgensi: 42% kandidat di request ketat bergrade 0, sedangkan di request santai hanya 8%. Grup training rata-rata berisi 3,65 kandidat lolos, dan 14,1% grup punya grade yang sama semua sehingga tidak memberi sinyal ranking.

In [10]:
cand_cat = candidates.merge(requests[["id", "material_id"]], left_on="request_id", right_on="id")
cand_cat = cand_cat.merge(materials[["id", "category"]].rename(columns={"id": "material_id"}), on="material_id")
cand_cat["reasons"] = cand_cat["filter_reasons"].fillna("[]").map(json.loads)

reason_names = {
    "melebihi_budget": "Melebihi budget",
    "tidak_tersedia": "Tidak tersedia",
    "moq_lebih_besar": "MOQ lebih besar dari qty",
    "kapasitas_kurang": "Kapasitas kurang",
}
exploded = cand_cat.explode("reasons").dropna(subset=["reasons"])
reason_share = (exploded.groupby(["category", "reasons"]).size() / cand_cat.groupby("category").size()).rename("proporsi")
reason_share = reason_share.reset_index()
reason_share["alasan"] = reason_share["reasons"].map(reason_names)

overall_fail = (~cand_cat["is_eligible"]).mean()
multi = (cand_cat["reasons"].map(len) > 1).mean()

fig = px.bar(
    reason_share, x="alasan", y="proporsi", color="category", barmode="group", color_discrete_map=CATEGORY_COLORS,
    labels={"alasan": "Alasan gugur", "proporsi": "Proporsi dari semua kandidat", "category": "Kategori"},
    title=f"Alasan kandidat gugur filter keras (total gugur {overall_fail:.1%} dari {len(cand_cat):,} kandidat)",
    category_orders={"alasan": list(reason_names.values())},
)
fig.update_layout(height=430, yaxis_tickformat=".0%")
fig.show()

print(f"kandidat gugur: {overall_fail:.1%}")
print(f"kandidat dengan lebih dari satu alasan: {multi:.1%}")
fail_by_category = (~cand_cat["is_eligible"]).groupby(cand_cat["category"]).mean()
print("kandidat gugur per kategori:", fail_by_category.map("{:.1%}".format).to_dict())

# konteks packaging, karakter tersembunyi dipakai hanya untuk analisis
price_by_category = traits.merge(vendors[["id", "category"]], left_on="vendor_id", right_on="id")
print("rata-rata price_level per kategori:",
      price_by_category.groupby("category")["price_level"].mean().round(2).to_dict())
p02 = materials.loc[materials["code"] == "P02", "id"].item()
p02_moq = offers.loc[offers["material_id"] == p02, "moq"]
p02_qty = requests.loc[requests["material_id"] == p02, "qty"].median()
print(f"P02: {(p02_moq >= 200).sum()} dari {len(p02_moq)} vendor punya MOQ >= 200, qty median request {p02_qty:.0f}")
(exploded["reasons"].value_counts() / len(cand_cat)).rename(index=reason_names).rename("proporsi").round(3)

kandidat gugur: 26.3%
kandidat dengan lebih dari satu alasan: 2.5%
kandidat gugur per kategori: {'bahan': '22.1%', 'jasa': '17.1%', 'packaging': '40.5%'}
rata-rata price_level per kategori: {'bahan': 1.02, 'jasa': 0.93, 'packaging': 1.12}
P02: 6 dari 9 vendor punya MOQ >= 200, qty median request 215


reasons
Melebihi budget             0.099
Tidak tersedia              0.090
MOQ lebih besar dari qty    0.063
Kapasitas kurang            0.036
Name: proporsi, dtype: float64

**Temuan:** Sebanyak 26,3% kandidat gugur filter keras, dengan alasan terbanyak melebihi budget (9,9%) dan tidak tersedia (9,0%). Packaging paling sering gugur (40,5%) karena vendor packaging di seed ini cenderung mahal (rata-rata `price_level` 1,12) dan MOQ kotak duplex (P02) 200 pcs di 6 dari 9 vendor, hampir sama dengan qty median request.

## Ringkasan Temuan dan Implikasi untuk Modeling

1. **Harga saja tidak cukup untuk memilih vendor.** Harga hampir tidak berkorelasi dengan keandalan, jadi model butuh fitur histori seperti `on_time_rate`, `avg_delay_days`, dan `defect_rate`.
2. **Janji lead time kurang bisa dipercaya saat musim ramai.** Fitur musim ramai dan `slack_days` perlu masuk ke model, terutama untuk request ber-priority `high`.
3. **Histori transaksi tidak acak.** Firaza lebih sering memilih vendor murah, sehingga vendor bagus yang jarang dipilih punya histori sedikit. Vendor 6, 16, dan 28 (cold start) hampir tanpa histori dan harus memakai nilai awal kategori.
4. **Konteks request sangat menentukan label.** Grade berubah besar antara request ketat dan santai, jadi fitur konteks seperti `priority`, `slack_days`, dan `qty / weekly_capacity` wajib ada.
5. **Grup packaging lebih kecil** karena banyak kandidat gugur filter. Evaluasi per kategori perlu dilaporkan terpisah.
6. **Periode test (Juli s.d. September 2026) hampir tanpa musim ramai.** Perilaku model saat musim ramai lebih banyak teruji di periode validasi dan di uji skenario.